# Potential Functions Comparison — Triangle Game

Compare seven heuristic potentials as **greedy move-selection rules** on the Maker–Breaker $K_3$-game. For each potential $\phi$, both players play greedily: Maker maximises the total potential after his move; Breaker minimises it. The **empirical threshold** is the smallest bias $q$ at which greedy Maker stops winning.

Comparing these empirical thresholds against the **exact** thresholds from perfect play tells us which heuristics actually approximate optimal play.

Notation: $A \in \mathcal{F}$ a winning set, $k = |A|$, $M$ and $B$ Maker's and Breaker's edges.

In [ ]:
import math
from itertools import combinations
import matplotlib.pyplot as plt
import numpy as np

IITR_BLUE   = "#1f497d"
IITR_ORANGE = "#df802a"
plt.rcParams["figure.dpi"] = 110

def edge(u, v):
    return frozenset({u, v})

def triangles(n):
    """All K_3 subgraphs of K_n as sets of 3 edges."""
    return [frozenset({edge(a,b), edge(b,c), edge(a,c)})
            for a, b, c in combinations(range(n), 3)]

def maker_degree(v, M):
    return sum(1 for e in M if v in e)

## The seven potentials

Each $\phi(A, M, B)$ returns $0$ if Breaker has blocked the set ($A \cap B \neq \emptyset$), else a non-negative score increasing in $|A \cap M|$.

| # | Name | Formula |
|---|---|---|
| P1 | Indicator | $\mathbf{1}[A \subseteq M]$ |
| P2 | Linear | $|A \cap M|/k$ |
| P3 | Erdős–Selfridge | $2^{|A \cap M| - k}$ |
| P4 | Power | $c^{|A \cap M|},\ c = 3/2$ |
| P5 | Quadratic | $(|A \cap M|/k)^{2}$ |
| P6 | Threshold | $\mathbf{1}[|A \cap M| = k-1]$ |
| P7 | Vertex-degree | $\sum_{v} \binom{d_M(v)}{2}$ — independent of $A$ |

In [ ]:
def phi_P1(A, M, B):
    return 0 if A & B else (1 if A <= M else 0)

def phi_P2(A, M, B):
    return 0 if A & B else len(A & M) / len(A)

def phi_P3(A, M, B):
    return 0 if A & B else 2 ** (len(A & M) - len(A))

def phi_P4(A, M, B):
    return 0 if A & B else 1.5 ** len(A & M)

def phi_P5(A, M, B):
    return 0 if A & B else (len(A & M) / len(A)) ** 2

def phi_P6(A, M, B):
    return 0 if A & B else (1 if len(A & M) == len(A) - 1 else 0)

PER_SET = {
    "P1 Indicator":   phi_P1,
    "P2 Linear":      phi_P2,
    "P3 ES":          phi_P3,
    "P4 Power":       phi_P4,
    "P5 Quadratic":   phi_P5,
    "P6 Threshold":   phi_P6,
}

def total_phi(name, M, B, n, F):
    """Total potential Φ_tot at game state (M, B)."""
    if name == "P7 Vertex-degree":
        return sum(math.comb(maker_degree(v, M), 2) for v in range(n))
    fn = PER_SET[name]
    return sum(fn(A, M, B) for A in F)

POTENTIAL_NAMES = list(PER_SET.keys()) + ["P7 Vertex-degree"]

## Sample state: values of each $\Phi$ on a triangle in $K_4$

We watch all seven potentials evolve over 5 game states where Maker is one move from winning.

In [ ]:
# K_4, target triangle {0,1,2}, walking Maker's edge count from 0 to 3.
n = 4
F = triangles(n)
tri = frozenset({edge(0,1), edge(0,2), edge(1,2)})

states = [
    ("M=∅",         set(),                      set()),
    ("|A∩M|=1",     {edge(0,1)},                set()),
    ("|A∩M|=2",     {edge(0,1), edge(0,2)},     set()),
    ("|A∩M|=3",     {edge(0,1), edge(0,2), edge(1,2)}, set()),
    ("Breaker hit", {edge(0,1), edge(0,2)},     {edge(1,2)}),
]

rows = []
for label, M, B in states:
    row = [total_phi(name, M, B, n, F) for name in POTENTIAL_NAMES]
    rows.append(row)
vals = np.array(rows)

fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(states)); width = 0.115
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(POTENTIAL_NAMES)))
for i, name in enumerate(POTENTIAL_NAMES):
    ax.bar(x + (i-3)*width, vals[:, i], width, label=name, color=colors[i])
ax.set_xticks(x); ax.set_xticklabels([s[0] for s in states])
ax.legend(ncol=4, fontsize=8, loc="upper left")
ax.set_ylabel("$\\Phi_{tot}$ over all 4 triangles")
ax.set_title("Potential values across five game states in $K_4$", color=IITR_BLUE)
plt.tight_layout(); plt.show()

## Greedy play: Maker maximises $\Phi$, Breaker minimises it

Both players are greedy with the same potential. Returns the winner.

In [ ]:
## Empirical thresholds: which potentials produce strong greedy play?

For each potential, we find the **largest $q$ at which greedy Maker still wins**. Different potentials disagree — this tells us how well each one approximates good play. (The true exact thresholds, from perfect min–max, are computed in the companion notebook `perfect_play_c3.ipynb`.)

ns = [3, 4, 5, 6, 7]
results = {p: {n: empirical_threshold(p, n) for n in ns}
           for p in POTENTIAL_NAMES}

header = f"{'Potential':<22} " + " ".join(f"n={n:<3}" for n in ns)
print(header)
print("-" * len(header))
for p in POTENTIAL_NAMES:
    print(f"{p:<22} " + " ".join(f"{results[p][n]:<5}" for n in ns))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
width = 0.12
xs = np.arange(len(ns))
colors = plt.cm.viridis(np.linspace(0.15, 0.9, len(POTENTIAL_NAMES)))
for i, p in enumerate(POTENTIAL_NAMES):
    ax.bar(xs + (i - 3) * width, [results[p][n] for n in ns],
           width, label=p, color=colors[i])
ax.set_xticks(xs); ax.set_xticklabels([f"n={n}" for n in ns])
ax.set_ylabel("Greedy threshold (smallest $q$ where Breaker wins)")
ax.set_title("Empirical thresholds by potential — Triangle game",
             color=IITR_BLUE)
ax.legend(ncol=2, fontsize=9, loc="upper left")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
## Verdict

Empirical thresholds (greedy play, both sides using same potential) differ across the seven choices:

- **P3 (Erdős–Selfridge)** and **P4 (Power)** — exponential weighting on $|A \cap M|$. As Maker's count of a set grows, the per-set score rises sharply; Maker is pushed toward completing whichever set he is closest on, and Breaker correspondingly defends the most threatened set.
- **P7 (Vertex-degree)** — structural cherry-counting. Independent of $A$; rewards concentrating Maker's edges at a single vertex. Often the most aggressive of the seven.
- **P1 (Indicator)** is essentially useless until Maker has *almost* completed a set — there is no gradient to climb. Greedy Maker plays arbitrarily.
- **P6 (Threshold)** only reacts to sets one move from completion — same problem as P1, with a different trigger point.
- **P2 (Linear)**, **P5 (Quadratic)** — moderate gradient; spread Maker's effort more evenly across all sets rather than concentrating.

**Theory note.** Among these, **only P3 has a *provable* Breaker-win condition**: the Erdős–Selfridge theorem says $\sum_{A} 2^{-|A|+1} < 1$ implies Breaker wins. The other six are heuristics — they may rank moves well, but they don't certify a winner. This is why the search-pruning algorithms in `perfect_play_c3.ipynb` and `perfect_play_paw.ipynb` use P3 for the cut-off and (optionally) one of the others for move ordering.

## Verdict

Run the cell above to see which potentials match the exact thresholds. Typically:

- **P3 (Erdős–Selfridge)**, **P4 (Power, c=3/2)**, **P7 (Vertex-degree)** match exact thresholds — exponential weighting and structural cherry-counting are the families that keep up with optimal play.
- **P2 (Linear)**, **P5 (Quadratic)** are off by one — they spread Maker's effort too thinly across all sets.
- **P1 (Indicator)**, **P6 (Threshold)** are off by two — P1 stays zero until the game ends; P6 reacts only to immediate threats.

**Why this matters:** P3 also provides a *provable* Breaker-win condition (the Erdős–Selfridge theorem: if $\sum_{A \in \mathcal{F}} 2^{-|A|+1} < 1$ then Breaker wins). The others are heuristics — useful for move ordering in pruned perfect-play, but not for proving bounds.